# Data Preprocessing
## Ethiopian Household Wealth Prediction

**Pipeline:**
1. Load cleaned data from EDA step
2. Handle missing values (median imputation)
3. Outlier treatment (winsorization)
4. Feature engineering (interactions, ratios)
5. Encode categorical variables
6. Scale numerical features
7. Split: Train (70%) / Validation (15%) / Test (15%)

**Why this pipeline:**
- Median imputation: Robust to outliers in survey data
- Winsorization at 3σ: Preserves data while limiting extreme values
- StandardScaler: Required for regularization models (Ridge/Lasso)
- Stratified-like split by wave: Ensures all time periods in each split


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import sys, os, joblib, warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

pd.set_option('display.max_columns', 60)
%matplotlib inline

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv('../data/processed/all_waves_clean.csv', low_memory=False)
print(f"Loaded: {df.shape[0]:,} households × {df.shape[1]} features")

# Drop pure ID columns
id_cols = [c for c in df.columns if c in ['hhid', 'household_id', 'household_id2', 'ea_id']]
df = df.drop(columns=[c for c in id_cols if c in df.columns])

# ============================================================
# 2. DEFINE TARGET
# ============================================================
# We predict LOG consumption (better distribution for regression)
target = 'log_total_consumption'

if target not in df.columns:
    # Create it if not present
    raw_target = [c for c in df.columns if 'total_consumption' in c.lower() and 'log' not in c.lower()]
    if raw_target:
        df[target] = np.log1p(df[raw_target[0]])
        print(f"Created log target from {raw_target[0]}")
    else:
        raise ValueError("No target variable found!")

print(f"Target: {target}")
print(f"  Mean: {df[target].mean():.3f}, Std: {df[target].std():.3f}")
print(f"  Skew: {df[target].skew():.2f} (close to 0 = good for regression)")

# Drop raw consumption (we use log version)
for c in df.columns:
    if 'total_consumption' in c.lower() and 'log' not in c.lower():
        df = df.drop(columns=[c])

# ============================================================
# 3. HANDLE MISSING VALUES
# ============================================================
# Drop rows where target is missing
before = len(df)
df = df.dropna(subset=[target])
print(f"Dropped {before - len(df):,} rows with missing target")

# Drop columns with >60% missing (too sparse to impute meaningfully)
missing_pct = df.isnull().sum() / len(df)
high_miss = missing_pct[missing_pct > 0.6].index.tolist()
if high_miss:
    df = df.drop(columns=high_miss)
    print(f"Dropped {len(high_miss)} columns with >60% missing")

# Separate features and target
X = df.drop(columns=[target])
y = df[target]

# Impute numeric: median (robust to outliers in survey data)
num_cols = X.select_dtypes(include=[np.number]).columns
if len(num_cols) > 0:
    imputer = SimpleImputer(strategy='median')
    X[num_cols] = imputer.fit_transform(X[num_cols])

# Impute categorical: most frequent
cat_cols = X.select_dtypes(include=['object']).columns
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown')

print(f"Remaining missing: {X.isnull().sum().sum()}")

# ============================================================
# 4. ENCODE CATEGORICAL VARIABLES
# ============================================================
# Label encode categorical columns
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print(f"All numeric: {len(X.select_dtypes(include=[np.number]).columns)} columns")

# ============================================================
# 5. REMOVE LOW-VARIANCE & HIGHLY CORRELATED FEATURES
# ============================================================
# Remove features with near-zero variance
selector = VarianceThreshold(threshold=0.001)
X_filtered = selector.fit_transform(X)
retained = X.columns[selector.get_support()].tolist()
dropped_var = len(X.columns) - len(retained)
X = pd.DataFrame(X_filtered, columns=retained, index=X.index)
print(f"Removed {dropped_var} low-variance features")

# Remove highly correlated features (>0.95)
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = [c for c in upper.columns if any(upper[c] > 0.95)]
if high_corr:
    X = X.drop(columns=high_corr)
    print(f"Removed {len(high_corr)} highly correlated features (r > 0.95)")

print(f"Final feature count: {X.shape[1]}")

# ============================================================
# 6. OUTLIER TREATMENT
# ============================================================
# Cap features at 5 standard deviations
for col in X.select_dtypes(include=[np.number]).columns:
    mean, std = X[col].mean(), X[col].std()
    if std > 0:
        X[col] = X[col].clip(mean - 5*std, mean + 5*std)

print("Features capped at μ ± 5σ")

# ============================================================
# 7. SCALE FEATURES
# ============================================================
# StandardScaler: zero mean, unit variance
# Required for Ridge/Lasso (regularization is scale-sensitive)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print(f"Scaled: mean≈{X_scaled.mean().mean():.6f}, std≈{X_scaled.std().mean():.6f}")

# ============================================================
# 8. TRAIN/VALIDATION/TEST SPLIT
# ============================================================
# Split preserving temporal distribution (15% test, 15% val)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42  # 15% of total
)

print(f"""
DATA SPLITS:
  Training:   {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X_scaled)*100:.1f}%)
  Validation: {X_val.shape[0]:,} samples ({X_val.shape[0]/len(X_scaled)*100:.1f}%)
  Test:       {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X_scaled)*100:.1f}%)
  Features:   {X_train.shape[1]}
""")

# ============================================================
# 9. SAVE PROCESSED DATA
# ============================================================
OUT = '../data/processed/'
os.makedirs(OUT, exist_ok=True)

X_train.to_csv(f'{OUT}X_train.csv', index=False)
X_val.to_csv(f'{OUT}X_val.csv', index=False)
X_test.to_csv(f'{OUT}X_test.csv', index=False)
y_train.to_csv(f'{OUT}y_train.csv', index=False, header=True)
y_val.to_csv(f'{OUT}y_val.csv', index=False, header=True)
y_test.to_csv(f'{OUT}y_test.csv', index=False, header=True)

# Full processed dataset (for reference)
full = X_scaled.copy()
full[target] = y.values
full.to_csv(f'{OUT}processed_data.csv', index=False)

# Save scaler for inference
joblib.dump(scaler, f'{OUT}scaler.pkl')
joblib.dump(X.columns.tolist(), f'{OUT}feature_names.pkl')

print("✓ All processed data saved to ../data/processed/")
print("  Ready for modeling → Open 03_modeling.ipynb")

Loaded: 238,562 households × 8 features
Target: log_total_consumption
  Mean: 11.047, Std: 0.865
  Skew: -0.18 (close to 0 = good for regression)
Dropped 374 rows with missing target
Remaining missing: 0
All numeric: 5 columns
Removed 0 low-variance features
Final feature count: 5
Features capped at μ ± 5σ
Scaled: mean≈0.000000, std≈1.000002

DATA SPLITS:
  Training:   166,826 samples (70.0%)
  Validation: 35,633 samples (15.0%)
  Test:       35,729 samples (15.0%)
  Features:   5

✓ All processed data saved to ../data/processed/
  Ready for modeling → Open 03_modeling.ipynb
